In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='euc-kr')

df = load_dataset('크루즈입항정보')

In [8]:
# ============================================================
# 1. 색상 설정
# ============================================================

BLUE = "#3478C0"    # 파란색
ORANGE = "#E67E22"  # 오렌지색
GRAY = "#657487"    # 회색


# ============================================================
# 2. 크루즈 입항 데이터 추출
# ============================================================

cruise = df[
    (df["입출"] == "입항") &
    (df["선박용도"] == "크루즈선")
].copy()

cruise["입항일시"] = pd.to_datetime(
    cruise["입항일시"],
    errors="coerce"
)

cruise = cruise.dropna(subset=["입항일시"]).copy()

cruise["연도"] = cruise["입항일시"].dt.year


# ============================================================
# 3. 연도별 입항 건수 집계
# ============================================================

annual = (
    cruise.groupby("연도")
    .size()
    .reset_index(name="입항건수")
)

# 전체 연도 범위 생성
all_years = range(
    int(annual["연도"].min()),
    int(annual["연도"].max()) + 1
)

# 누락된 연도 추가
annual = (
    annual.set_index("연도")
    .reindex(all_years)
    .rename_axis("연도")
    .reset_index()
)

# 실제 무입항으로 확인한 연도는 0건으로 처리
annual["입항건수"] = annual["입항건수"].fillna(0)

# 연도 표시
annual["연도표시"] = annual["연도"].apply(
    lambda year: "2026<br>(누계)" if year == 2026 else str(year)
)


# ============================================================
# 4. 막대 색상 설정
# ============================================================

# 2016년과 2025년 최댓값 강조
highlight_years = [2016, 2025]

colors = [
    GRAY if year == 2026
    else ORANGE if year in highlight_years
    else BLUE
    for year in annual["연도"]
]


# ============================================================
# 5. 막대 위 숫자 표시
# ============================================================

labels = [
    f"{value:,.0f}" if value > 0 else ""
    for value in annual["입항건수"]
]


# ============================================================
# 6. 그래프 생성
# ============================================================

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=annual["연도표시"],
        y=annual["입항건수"],

        marker_color=colors,
        marker_line_width=0,

        text=labels,
        textposition="outside",

        textfont=dict(
            size=20,
            color="#243247"
        ),

        cliponaxis=False,

        customdata=annual[["연도"]].to_numpy(),

        hovertemplate=(
            "<b>%{customdata[0]:.0f}년</b>"
            "<br>입항건수: %{y:,.0f}건"
            "<extra></extra>"
        )
    )
)


# ============================================================
# 7. 제목 및 레이아웃
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "<b>부산항 크루즈 연도별 입항건수</b>"
            "<br>"
            f"<span style='font-size:18px;color:{GRAY}'>"
            "연도별 선박 입항 규모의 장기 변화"
            "</span>"
        ),
        x=0.05,
        y=0.95,
        xanchor="left"
    ),

    template="plotly_white",

    width=1200,
    height=675,

    paper_bgcolor="white",
    plot_bgcolor="white",

    font=dict(
        family="Malgun Gothic, Arial, sans-serif",
        size=18,
        color="#243247"
    ),

    margin=dict(
        l=90,
        r=45,
        t=125,
        b=125
    ),

    bargap=0.32,
    showlegend=False
)


# ============================================================
# 8. X축 설정
# ============================================================

fig.update_xaxes(
    title=None,

    type="category",

    categoryorder="array",
    categoryarray=annual["연도표시"].tolist(),

    showgrid=False,

    tickfont=dict(size=17),

    fixedrange=True
)


# ============================================================
# 9. Y축 설정
# ============================================================

fig.update_yaxes(
    title="입항건수 (건)",

    tickformat=",d",

    range=[
        0,
        annual["입항건수"].max() * 1.20
    ],

    gridcolor="#E9EDF2",
    zeroline=False,

    fixedrange=True
)


# ============================================================
# 10. 코로나19 영향 구간 표시
# ============================================================

year_list = annual["연도"].tolist()

if 2020 in year_list and 2022 in year_list:

    start = year_list.index(2020)
    end = year_list.index(2022)

    fig.add_vrect(
        x0=start - 0.5,
        x1=end + 0.5,

        fillcolor="#E85D5D",
        opacity=0.12,

        line_width=0,
        layer="below"
    )

    fig.add_annotation(
        x=(start + end) / 2,

        y=0.85,

        xref="x",
        yref="paper",

        text="<b>코로나19 영향 구간</b><br>(2020~2022년)",

        showarrow=False,
        align="center",

        font=dict(
            family="Malgun Gothic",
            size=18,
            color="#C62828"
        )
    )


# ============================================================
# 11. 그래프 하단 주석
# ============================================================

fig.add_annotation(
    x=0,
    y=-0.20,

    xref="paper",
    yref="paper",

    text=(
        "※ 2026년은 10월 16일까지 누계로, 이전 연도의 연간 실적과 직접 비교 시 주의"
        "<br>※ 실제 무입항으로 확인된 연도는 0건으로 처리"
    ),

    showarrow=False,

    xanchor="left",
    align="left",

    font=dict(
        size=14,
        color=GRAY
    )
)


# ============================================================
# 12. 그래프 출력 및 이미지 저장 설정
# ============================================================

fig.show(
    config={
        "displaylogo": False,

        "toImageButtonOptions": {
            "format": "png",
            "filename": "PPT_연도별_입항건수",
            "width": 1200,
            "height": 675,
            "scale": 3
        }
    }
)